# Ordered Logistic Regression Results: FAIR² Dataset Exploration with `mlcroissant`
This notebook demonstrates how to explore the FAIR² dataset ([Kamadi et al., 2026](https://sen.science/doi/10.71728/senscience.y7m0-f273)), using the [mlcroissant](https://github.com/mlcommons/croissant) library. We follow best practices for referencing entities by their `@id` fields and perform step-by-step data loading, overview, extraction, and analysis.

### Dataset Source
- Croissant schema URL: [https://sen.science/doi/10.71728/senscience.y7m0-f273/fair2.json](https://sen.science/doi/10.71728/senscience.y7m0-f273/fair2.json)


In [ ]:
# Ensure `mlcroissant` library is installed
!pip install -q mlcroissant

## 1. Data Loading
Load dataset metadata and records using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the dataset URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.y7m0-f273/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata

print(f"{metadata.name}: {metadata.description}")

## 2. Data Overview
Let's review available record sets and their fields.

**Note**: In the Croissant schema, all entities (record sets, fields, columns) should be referenced by their `@id` fields.

In [ ]:
# Print all available record set @ids and their fields
record_sets = dataset.record_sets
print('Available record sets:')
for rs in record_sets:
    print(f"@id: {rs['@id']} | name: {rs.get('name', '[no name]')}")
    print('  Fields:')
    for field in rs.get('field', []):
        # Many fields have either dict or str
        if isinstance(field, dict):
            print(f"    - @id: {field.get('@id')} | name: {field.get('name', '[no name]')}")
        else:
            print(f"    - @id: {field}")
    print('----')
# Save record_set_ids for following cells
record_set_ids = [rs['@id'] for rs in record_sets]

### Load and preview sample records from each record set

Here we check what kind of records are available for each record set.

In [ ]:
for record_set_id in record_set_ids:
    print(f'Records in record set: {record_set_id}')
    try:
        records = list(dataset.records(record_set=record_set_id))
        if records:
            # Print only first 2 records for brevity
            for rec in records[:2]:
                print(rec)
            print(f"Total records: {len(records)}")
        else:
            print('No records found in this set.')
    except Exception as e:
        print(f'Error accessing records: {e}')
    print('----\n')

## 3. Data Extraction

Let's load data from the primary record set(s) into pandas DataFrames for analysis.

**Record sets to be loaded** (by `@id`):

In [ ]:
# Extract records from all record sets
dataframes = {}
for record_set_id in record_set_ids:
    try:
        records = list(dataset.records(record_set=record_set_id))
        if records:
            df = pd.DataFrame(records)
            dataframes[record_set_id] = df
            print(f"Loaded DataFrame for record set: {record_set_id}")
            print(f"Columns: {df.columns.tolist()}")
            display(df.head())
        else:
            print(f"No records to extract for record set: {record_set_id}")
    except Exception as e:
        print(f"Failed to extract data from {record_set_id}: {e}")

## 4. Exploratory Data Analysis (EDA)

We select a numeric field from one of the record sets, filter records based on a threshold, normalize the field, and optionally group by a categorical field (if available).

Let's automatically search for an appropriate numeric field to demonstrate this.

In [ ]:
# Choose a record set with data for EDA
rs_id_for_eda = None
df_for_eda = None
for k, v in dataframes.items():
    if len(v.columns) > 0 and v.select_dtypes('number').shape[1] > 0:
        rs_id_for_eda = k
        df_for_eda = v
        break

if rs_id_for_eda is None:
    raise ValueError('No numeric fields found in record sets.')

print(f'Using record set: {rs_id_for_eda} for EDA.')
numeric_fields = df_for_eda.select_dtypes('number').columns.tolist()
print(f'Available numeric fields: {numeric_fields}')
numeric_field = numeric_fields[0] if numeric_fields else None
if numeric_field is None:
    raise ValueError('No numeric field available for EDA.')

# Filter
threshold = df_for_eda[numeric_field].mean()
filtered_df = df_for_eda[df_for_eda[numeric_field] > threshold].copy()
print(f"Filtered records with {numeric_field} > {threshold:.2f} (mean):")
display(filtered_df.head())

# Normalize
filtered_df[f"{numeric_field}_normalized"] = (
    (filtered_df[numeric_field] - filtered_df[numeric_field].mean()) / filtered_df[numeric_field].std()
)
print(f"Normalized {numeric_field}:")
display(filtered_df[[numeric_field, f"{numeric_field}_normalized"]].head())

# Try grouping by an available categorical field (if any non-numeric columns exist)
group_fields = [col for col in filtered_df.columns if filtered_df[col].dtype == 'object' and col != numeric_field]
group_field = group_fields[0] if group_fields else None
if group_field:
    grouped_df = filtered_df.groupby(group_field)[numeric_field].mean()
    print(f"Grouped mean of {numeric_field} by {group_field}:")
    display(grouped_df.head())
else:
    print('No categorical field found to group by in this record set.')

## 5. Visualization

Now let's visualize the distribution of the selected numeric field (and its normalized version) in this record set.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

plt.figure(figsize=(10, 5))
sns.histplot(filtered_df[numeric_field], kde=True, bins=20, color='skyblue')
plt.title(f'Distribution of {numeric_field} (Filtered)')
plt.xlabel(numeric_field)
plt.ylabel('Frequency')
plt.show()

if f"{numeric_field}_normalized" in filtered_df.columns:
    plt.figure(figsize=(10, 5))
    sns.histplot(filtered_df[f"{numeric_field}_normalized"], kde=True, bins=20, color='orange')
    plt.title(f'Normalized {numeric_field} (Filtered)')
    plt.xlabel(f"{numeric_field}_normalized")
    plt.ylabel('Frequency')
    plt.show()

## 6. Conclusion

- We loaded and inspected metadata and record sets from the FAIR² dataset using `mlcroissant`.
- Referenced all fields and record sets strictly by their `@id` as per Croissant best practice.
- Performed EDA on available numeric fields, including filtering, normalization, grouping, and visualization.
- This notebook provides a template for further analysis, easily extensible for deeper statistical or machine learning analyses.

**For more details, see the [schema documentation](https://sen.science/doi/10.71728/senscience.y7m0-f273/fair2.json) and the [mlcroissant reference](https://github.com/mlcommons/croissant).**